In [ ]:
!pip install -q diffusers accelerate transformers kagglehub bitsandbytes sentencepiece protobuf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 11.6 MB/s eta 0:00:00


In [ ]:
from huggingface_hub import notebook_login
notebook_login()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [ ]:
import os, gc, math, random, logging
from pathlib import Path

import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
from tqdm.auto import tqdm

from accelerate import Accelerator
from accelerate.logging import get_logger
from accelerate.utils import ProjectConfiguration

from transformers import (
    CLIPTokenizer, CLIPTextModelWithProjection,
    T5TokenizerFast, T5EncoderModel,
    get_cosine_schedule_with_warmup,
)
from diffusers import (
    SD3Transformer2DModel,
    FlowMatchEulerDiscreteScheduler,
    AutoencoderKL,
    StableDiffusion3Pipeline,
)
import kagglehub

Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.


In [ ]:
MODEL_ID            = "stabilityai/stable-diffusion-3-medium-diffusers"
OUTPUT_DIR          = "/content/sd3-finetuned"
KAGGLE_DATASET      = "adityajn105/flickr8k"
NUM_IMAGES          = 10
RESOLUTION          = 256
TRAIN_BATCH_SIZE    = 1
GRADIENT_ACCUM      = 8
NUM_EPOCHS          = 10
LEARNING_RATE       = 5e-6
LR_WARMUP_STEPS     = 40
MIXED_PRECISION     = "fp16"
GRADIENT_CKPT       = True
SAVE_EVERY_N_EPOCHS = 2
SEED                = 42
MAX_GRAD_NORM       = 1.0
USE_8BIT_ADAM       = True

logging.basicConfig(level=logging.INFO)
logger = get_logger(__name__, log_level="INFO")

In [ ]:
def free_memory(*objects):
    """Delete objects and flush GPU cache."""
    for obj in objects:
        del obj
    gc.collect()
    torch.cuda.empty_cache()


def vram_usage(label=""):
    used = torch.cuda.memory_allocated() / 1e9
    reserved = torch.cuda.memory_reserved() / 1e9
    print(f"[VRAM {label}] allocated {used:.2f} GB | reserved {reserved:.2f} GB")

In [ ]:
class EmbeddingDataset(Dataset):
    """
    Serves pre-computed (latent, prompt_embeds, pooled_embeds) tensors.
    After pre-computation the frozen encoders are deleted from GPU entirely,
    so the training loop only ever touches the transformer.
    """
    def __init__(self, latents, prompt_embeds, pooled_embeds):
        self.latents        = latents
        self.prompt_embeds  = prompt_embeds
        self.pooled_embeds  = pooled_embeds

    def __len__(self):
        return len(self.latents)

    def __getitem__(self, idx):
        return {
            "latents":       self.latents[idx],
            "prompt_embeds": self.prompt_embeds[idx],
            "pooled_embeds": self.pooled_embeds[idx],
        }


def load_kaggle_dataset(dataset_name, num_images, seed):
    logger.info(f"Downloading Kaggle dataset: {dataset_name}")
    dataset_path = kagglehub.dataset_download(dataset_name)
    base = Path(dataset_path)

    image_exts = {".jpg", ".jpeg", ".png", ".webp"}
    all_images = sorted(p for p in base.rglob("*") if p.suffix.lower() in image_exts)
    logger.info(f"Found {len(all_images)} images.")

    random.seed(seed)
    selected = random.sample(all_images, min(num_images, len(all_images)))

    # ── captions ─────────────────────────────────────────────────────────────
    caption_map = {}
    for cap_file in base.rglob("captions.txt"):
        with open(cap_file, encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if not line or line.lower().startswith("image"):
                    continue
                parts = line.split("\t", 1)
                if len(parts) == 2:
                    key = parts[0].split("#")[0].strip()
                    if key not in caption_map:
                        caption_map[key] = parts[1].strip()
        break

    if not caption_map:
        import csv
        for cap_file in base.rglob("*.csv"):
            with open(cap_file, newline="", encoding="utf-8") as f:
                reader = csv.DictReader(f)
                for row in reader:
                    img_col = next((k for k in row if "image" in k.lower() or "file" in k.lower()), None)
                    cap_col = next((k for k in row if "caption" in k.lower() or "comment" in k.lower()
                                    or "description" in k.lower()), None)
                    if img_col and cap_col:
                        key = Path(row[img_col]).name
                        if key not in caption_map:
                            caption_map[key] = row[cap_col].strip()
            if caption_map:
                break

    image_paths, captions = [], []
    for p in selected:
        cap = caption_map.get(p.name) or caption_map.get(p.stem) or f"a photo of {p.stem.replace('_',' ')}"
        image_paths.append(str(p))
        captions.append(cap)

    logger.info(f"Using {len(image_paths)} pairs.")
    for i in range(min(3, len(captions))):
        logger.info(f"  [{i}] {Path(image_paths[i]).name} → \"{captions[i][:80]}\"")
    return image_paths, captions


In [ ]:
IMG_TRANSFORM = transforms.Compose([
    transforms.Resize(RESOLUTION, interpolation=transforms.InterpolationMode.BILINEAR),
    transforms.CenterCrop(RESOLUTION),
    transforms.ToTensor(),
    transforms.Normalize([0.5], [0.5]),
])


@torch.no_grad()
def precompute_embeddings(image_paths, captions, device):
    """
    Encode all images (VAE) and captions (CLIP×2 + T5) to tensors,
    store on CPU. Returns lists ready to build EmbeddingDataset.
    """
    weight_dtype = torch.float16   # T4 is fp16 only

    # ── load frozen encoders ─────────────────────────────────────────────────
    logger.info("Loading VAE …")
    vae = AutoencoderKL.from_pretrained(MODEL_ID, subfolder="vae",
                                        torch_dtype=weight_dtype).to(device)
    vae.eval()
    vram_usage("after VAE load")

    logger.info("Loading CLIP encoders …")
    clip_tok_l  = CLIPTokenizer.from_pretrained(MODEL_ID, subfolder="tokenizer")
    clip_tok_g  = CLIPTokenizer.from_pretrained(MODEL_ID, subfolder="tokenizer_2")
    clip_enc_l  = CLIPTextModelWithProjection.from_pretrained(
                        MODEL_ID, subfolder="text_encoder",
                        torch_dtype=weight_dtype).to(device)
    clip_enc_g  = CLIPTextModelWithProjection.from_pretrained(
                        MODEL_ID, subfolder="text_encoder_2",
                        torch_dtype=weight_dtype).to(device)
    clip_enc_l.eval(); clip_enc_g.eval()
    vram_usage("after CLIP load")

    # T5-XXL stays on CPU — too large for T4 alongside the transformer
    logger.info("Loading T5 on CPU (large model — intentionally kept off GPU) …")
    t5_tok = T5TokenizerFast.from_pretrained(MODEL_ID, subfolder="tokenizer_3")
    t5_enc = T5EncoderModel.from_pretrained(MODEL_ID, subfolder="text_encoder_3",
                                             torch_dtype=weight_dtype)   # CPU
    t5_enc.eval()

    all_latents, all_prompt_embeds, all_pooled = [], [], []

    for img_path, caption in tqdm(zip(image_paths, captions),
                                   total=len(image_paths), desc="Pre-computing"):
        # ── VAE ──────────────────────────────────────────────────────────────
        img = Image.open(img_path).convert("RGB")
        pixel = IMG_TRANSFORM(img).unsqueeze(0).to(device, dtype=weight_dtype)
        latent = vae.encode(pixel).latent_dist.sample() * vae.config.scaling_factor
        all_latents.append(latent.squeeze(0).cpu())

        # ── CLIP-L ───────────────────────────────────────────────────────────
        def clip_encode(tok, enc, text):
            toks = tok(text, padding="max_length", max_length=77,
                        truncation=True, return_tensors="pt").to(device)
            out  = enc(**toks, output_hidden_states=True)
            return out.hidden_states[-2], out.text_embeds   # [1,77,D], [1,D]

        h_l, p_l = clip_encode(clip_tok_l, clip_enc_l, caption)
        h_g, p_g = clip_encode(clip_tok_g, clip_enc_g, caption)
        clip_hidden = torch.cat([h_l, h_g], dim=-1)   # [1,77,2048]

        # ── T5 (CPU inference, result moved to CPU store) ────────────────────
        t5_toks = t5_tok(caption, padding="max_length", max_length=256,
                          truncation=True, return_tensors="pt")   # stays on CPU
        t5_hidden = t5_enc(**t5_toks).last_hidden_state            # [1,256,4096] CPU

        # pad CLIP to 4096 and concat along sequence dim
        clip_pad = F.pad(clip_hidden.cpu(), (0, 4096 - clip_hidden.shape[-1]))  # [1,77,4096]
        prompt_embeds = torch.cat([clip_pad, t5_hidden], dim=1)    # [1,333,4096]
        pooled_embeds = torch.cat([p_l, p_g], dim=-1)              # [1,2048]

        all_prompt_embeds.append(prompt_embeds.squeeze(0).cpu())
        all_pooled.append(pooled_embeds.squeeze(0).cpu())

    # ── free all frozen encoders from GPU ────────────────────────────────────
    logger.info("Freeing encoders from GPU …")
    free_memory(vae, clip_enc_l, clip_enc_g, t5_enc,
                clip_tok_l, clip_tok_g, t5_tok)
    vram_usage("after encoder cleanup")

    return all_latents, all_prompt_embeds, all_pooled



In [ ]:
def train():
    torch.manual_seed(SEED)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # accelerator
    project_cfg = ProjectConfiguration(project_dir=OUTPUT_DIR,
                                        logging_dir=os.path.join(OUTPUT_DIR, "logs"))
    accelerator = Accelerator(
        gradient_accumulation_steps=GRADIENT_ACCUM,
        mixed_precision=MIXED_PRECISION,
        project_config=project_cfg,
    )

    logger.info(f"Device: {device}")
    vram_usage("start")

    # dataset
    image_paths, captions = load_kaggle_dataset(KAGGLE_DATASET, NUM_IMAGES, SEED)

    # Pre-compute everything so encoders don't compete for VRAM during training
    latents, prompt_embeds, pooled_embeds = precompute_embeddings(image_paths, captions, device)

    dataset    = EmbeddingDataset(latents, prompt_embeds, pooled_embeds)
    dataloader = DataLoader(dataset, batch_size=TRAIN_BATCH_SIZE,
                             shuffle=True, num_workers=0, pin_memory=False)

    os.makedirs(OUTPUT_DIR, exist_ok=True)

    # transformer (trainable)
    logger.info("Loading SD3 transformer …")
    transformer = SD3Transformer2DModel.from_pretrained(
        MODEL_ID, subfolder="transformer",
        torch_dtype=torch.float16,
    )
    transformer.requires_grad_(True)   # full finetune — all params

    if GRADIENT_CKPT:
        transformer.enable_gradient_checkpointing()

    vram_usage("after transformer load")

    # noise scheduler
    noise_scheduler = FlowMatchEulerDiscreteScheduler.from_pretrained(
        MODEL_ID, subfolder="scheduler"
    )

    # optimizer
    if USE_8BIT_ADAM:
        try:
            import bitsandbytes as bnb
            optimizer = bnb.optim.AdamW8bit(
                transformer.parameters(), lr=LEARNING_RATE,
                betas=(0.9, 0.999), weight_decay=1e-2, eps=1e-8,
            )
            logger.info("Using 8-bit AdamW (bitsandbytes)")
        except ImportError:
            logger.warning("bitsandbytes not found — falling back to fp32 AdamW")
            optimizer = torch.optim.AdamW(
                transformer.parameters(), lr=LEARNING_RATE,
                betas=(0.9, 0.999), weight_decay=1e-2, eps=1e-8,
            )
    else:
        optimizer = torch.optim.AdamW(
            transformer.parameters(), lr=LEARNING_RATE,
            betas=(0.9, 0.999), weight_decay=1e-2, eps=1e-8,
        )

    num_update_steps = math.ceil(len(dataloader) / GRADIENT_ACCUM) * NUM_EPOCHS
    lr_scheduler = get_cosine_schedule_with_warmup(
        optimizer,
        num_warmup_steps=LR_WARMUP_STEPS,
        num_training_steps=num_update_steps,
    )

    transformer, optimizer, dataloader, lr_scheduler = accelerator.prepare(
        transformer, optimizer, dataloader, lr_scheduler
    )
    vram_usage("after accelerate prepare")

    # training loop
    logger.info("━━━━━  Starting Training  ━━━━━")
    logger.info(f"  Model            : {MODEL_ID}")
    logger.info(f"  Dataset size     : {len(dataset)}")
    logger.info(f"  Resolution       : {RESOLUTION}×{RESOLUTION}")
    logger.info(f"  Batch / Accum    : {TRAIN_BATCH_SIZE} / {GRADIENT_ACCUM}")
    logger.info(f"  Epochs           : {NUM_EPOCHS}")
    logger.info(f"  Learning rate    : {LEARNING_RATE}")
    logger.info(f"  Mixed precision  : {MIXED_PRECISION}")
    logger.info(f"  Gradient ckpt    : {GRADIENT_CKPT}")
    logger.info(f"  8-bit Adam       : {USE_8BIT_ADAM}")

    global_step = 0

    for epoch in range(NUM_EPOCHS):
        transformer.train()
        epoch_loss = 0.0

        pbar = tqdm(total=len(dataloader), desc=f"Epoch {epoch+1}/{NUM_EPOCHS}")

        for step, batch in enumerate(dataloader):
            with accelerator.accumulate(transformer):

                # move pre-computed tensors to GPU
                latents_b       = batch["latents"].to(device, dtype=torch.float16)
                prompt_embeds_b = batch["prompt_embeds"].to(device, dtype=torch.float16)
                pooled_embeds_b = batch["pooled_embeds"].to(device, dtype=torch.float16)

                bsz = latents_b.shape[0]

                # timestep + noise
                timesteps = torch.randint(
                    0, noise_scheduler.config.num_train_timesteps,
                    (bsz,), device=device,
                ).long()

                noise         = torch.randn_like(latents_b)
                noisy_latents = noise_scheduler.scale_noise(latents_b, timesteps, noise)

                # forward
                model_pred = transformer(
                    hidden_states         = noisy_latents,
                    timestep              = timesteps,
                    encoder_hidden_states = prompt_embeds_b,
                    pooled_projections    = pooled_embeds_b,
                    return_dict           = False,
                )[0]

                # Flow-matching target: velocity = noise − latent
                target = noise - latents_b
                loss   = F.mse_loss(model_pred.float(), target.float(), reduction="mean")

                epoch_loss += loss.detach().item()
                accelerator.backward(loss)

                if accelerator.sync_gradients:
                    accelerator.clip_grad_norm_(transformer.parameters(), MAX_GRAD_NORM)

                optimizer.step()
                lr_scheduler.step()
                optimizer.zero_grad(set_to_none=True)

            if accelerator.sync_gradients:
                global_step += 1

            pbar.update(1)
            pbar.set_postfix(loss=f"{loss.detach().item():.4f}")

        pbar.close()
        avg = epoch_loss / len(dataloader)
        logger.info(f"Epoch {epoch+1}/{NUM_EPOCHS} — avg loss: {avg:.4f}")
        vram_usage(f"end epoch {epoch+1}")

        # checkpoint
        if (epoch + 1) % SAVE_EVERY_N_EPOCHS == 0:
            ckpt = os.path.join(OUTPUT_DIR, f"checkpoint-epoch-{epoch+1}")
            accelerator.unwrap_model(transformer).save_pretrained(ckpt)
            logger.info(f"Saved checkpoint → {ckpt}")

    # save full pipeline
    accelerator.wait_for_everyone()
    logger.info("Saving final pipeline …")
    pipeline = StableDiffusion3Pipeline.from_pretrained(
        MODEL_ID,
        transformer=accelerator.unwrap_model(transformer),
        torch_dtype=torch.float16,
    )
    pipeline.save_pretrained(OUTPUT_DIR)
    logger.info(f"Done! Pipeline saved to {OUTPUT_DIR}")
    accelerator.end_training()

In [ ]:
def generate_samples(prompt: str, num_images: int = 2, out_dir: str = "/content/samples"):
    """
    Load the fine-tuned pipeline and generate samples.
    Runs with enable_model_cpu_offload() to stay within 16 GB.
    """
    import matplotlib.pyplot as plt

    pipeline = StableDiffusion3Pipeline.from_pretrained(
        OUTPUT_DIR, torch_dtype=torch.float16
    )
    # CPU offload: moves model layers to GPU one at a time — safe on T4
    pipeline.enable_model_cpu_offload()

    os.makedirs(out_dir, exist_ok=True)
    images = pipeline(
        prompt,
        num_images_per_prompt=num_images,
        num_inference_steps=28,
        height=RESOLUTION,
        width=RESOLUTION,
    ).images

    cols = num_images
    fig, axes = plt.subplots(1, cols, figsize=(4 * cols, 4))
    if cols == 1:
        axes = [axes]
    for i, (img, ax) in enumerate(zip(images, axes)):
        ax.imshow(img); ax.axis("off"); ax.set_title(f"#{i+1}")
        img.save(os.path.join(out_dir, f"sample_{i+1}.png"))

    plt.suptitle(prompt, wrap=True)
    plt.tight_layout()
    plt.savefig(os.path.join(out_dir, "grid.png"))
    plt.show()
    print(f"Saved {num_images} image(s) to {out_dir}")

In [ ]:
if __name__ == "__main__":
    train()

INFO:__main__:Device: cuda
INFO:__main__:Downloading Kaggle dataset: adityajn105/flickr8k


[VRAM start] allocated 0.00 GB | reserved 0.00 GB
Using Colab cache for faster access to the 'flickr8k' dataset.


INFO:__main__:Found 8091 images.
INFO:__main__:Using 10 pairs.
INFO:__main__:  [0] 3354489242_dd529ffa1f.jpg → "a photo of 3354489242 dd529ffa1f"
INFO:__main__:  [1] 2071309418_1d7580b0f0.jpg → "a photo of 2071309418 1d7580b0f0"
INFO:__main__:  [2] 1247181182_35cabd76f3.jpg → "a photo of 1247181182 35cabd76f3"
INFO:__main__:Loading VAE …
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:202: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(
INFO:httpx:HTTP Request: HEAD https://huggingface.co/stabilityai/stable-diffusion-3-medium-diffusers/resolve/main/vae/config.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET https://huggingface.co/stabilityai/stable-diffusion-3-medium-diffusers/resolve/main/vae/config.json "HTTP/1.1 200 OK"


config.json:   0%|          | 0.00/739 [00:00<?, ?B/s]

INFO:httpx:HTTP Request: HEAD https://huggingface.co/stabilityai/stable-diffusion-3-medium-diffusers/resolve/ea42f8cef0f178587cf766dc8129abd379c90671/vae/diffusion_pytorch_model.safetensors.index.json "HTTP/1.1 404 Not Found"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/stabilityai/stable-diffusion-3-medium-diffusers/resolve/ea42f8cef0f178587cf766dc8129abd379c90671/vae/diffusion_pytorch_model.safetensors "HTTP/1.1 302 Found"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/models/stabilityai/stable-diffusion-3-medium-diffusers/xet-read-token/ea42f8cef0f178587cf766dc8129abd379c90671 "HTTP/1.1 200 OK"


...

scheduler_config.json:   0%|          | 0.00/141 [00:00<?, ?B/s]

INFO:__main__:Using 8-bit AdamW (bitsandbytes)
INFO:__main__:━━━━━  Starting Training  ━━━━━
INFO:__main__:  Model            : stabilityai/stable-diffusion-3-medium-diffusers
INFO:__main__:  Dataset size     : 10
INFO:__main__:  Resolution       : 256×256
INFO:__main__:  Batch / Accum    : 1 / 8
INFO:__main__:  Epochs           : 10
INFO:__main__:  Learning rate    : 5e-06
INFO:__main__:  Mixed precision  : fp16
INFO:__main__:  Gradient ckpt    : True
INFO:__main__:  8-bit Adam       : True


[VRAM after accelerate prepare] allocated 5.67 GB | reserved 5.90 GB


Epoch 1/10:   0%|          | 0/10 [00:00<?, ?it/s]

IndexError: index 0 is out of bounds for dimension 0 with size 0

## More Optimizations


In [ ]:
import os, gc, math, random, logging
from pathlib import Path

import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
from tqdm.auto import tqdm

from accelerate import Accelerator
from accelerate.logging import get_logger
from accelerate.utils import ProjectConfiguration

from transformers import (
    CLIPTokenizer, CLIPTextModelWithProjection,
    T5TokenizerFast, T5EncoderModel,
    get_cosine_schedule_with_warmup,
)
from diffusers import (
    SD3Transformer2DModel,
    FlowMatchEulerDiscreteScheduler,
    AutoencoderKL,
    StableDiffusion3Pipeline,
)


Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.


In [ ]:
def add_flow_noise(latents: torch.Tensor, noise: torch.Tensor,
                   timesteps: torch.Tensor, num_train_timesteps: int = 1000) -> torch.Tensor:
    """
    SD3 uses Rectified Flow / Flow Matching.
    The forward process is a simple linear interpolation:
        x_t = (1 - t) * x_0  +  t * noise      where t ∈ [0, 1]

    `timesteps` are integers in [0, num_train_timesteps).
    We convert to t ∈ [0, 1] and broadcast over (B, C, H, W).

    This replaces `noise_scheduler.scale_noise()` which requires the
    scheduler to be initialised with set_timesteps() first (inference-only).
    """
    # t shape: [B, 1, 1, 1]
    t = (timesteps.float() / num_train_timesteps).view(-1, 1, 1, 1).to(latents.device)
    return (1.0 - t) * latents + t * noise
import kagglehub

In [ ]:
MODEL_ID            = "stabilityai/stable-diffusion-3-medium-diffusers"
OUTPUT_DIR          = "/content/sd3-finetuned"
KAGGLE_DATASET      = "adityajn105/flickr8k"
NUM_IMAGES          = 2
RESOLUTION          = 256
TRAIN_BATCH_SIZE    = 1
GRADIENT_ACCUM      = 8
NUM_EPOCHS          = 10
LEARNING_RATE       = 5e-6
LR_WARMUP_STEPS     = 40
MIXED_PRECISION     = "fp16"
GRADIENT_CKPT       = True
SAVE_EVERY_N_EPOCHS = 2
SEED                = 42
MAX_GRAD_NORM       = 1.0
USE_8BIT_ADAM       = True

logging.basicConfig(level=logging.INFO)
logger = get_logger(__name__, log_level="INFO")

In [ ]:
def free_memory(*objects):
    """Delete objects and flush GPU cache."""
    for obj in objects:
        del obj
    gc.collect()
    torch.cuda.empty_cache()


def vram_usage(label=""):
    used = torch.cuda.memory_allocated() / 1e9
    reserved = torch.cuda.memory_reserved() / 1e9
    print(f"[VRAM {label}] allocated {used:.2f} GB | reserved {reserved:.2f} GB")

In [ ]:
class EmbeddingDataset(Dataset):
    """
    Serves pre-computed (latent, prompt_embeds, pooled_embeds) tensors.
    After pre-computation the frozen encoders are deleted from GPU entirely,
    so the training loop only ever touches the transformer.
    """
    def __init__(self, latents, prompt_embeds, pooled_embeds):
        self.latents        = latents        # list of tensors on CPU
        self.prompt_embeds  = prompt_embeds
        self.pooled_embeds  = pooled_embeds

    def __len__(self):
        return len(self.latents)

    def __getitem__(self, idx):
        return {
            "latents":       self.latents[idx],
            "prompt_embeds": self.prompt_embeds[idx],
            "pooled_embeds": self.pooled_embeds[idx],
        }


def load_kaggle_dataset(dataset_name, num_images, seed):
    logger.info(f"Downloading Kaggle dataset: {dataset_name}")
    dataset_path = kagglehub.dataset_download(dataset_name)
    base = Path(dataset_path)

    image_exts = {".jpg", ".jpeg", ".png", ".webp"}
    all_images = sorted(p for p in base.rglob("*") if p.suffix.lower() in image_exts)
    logger.info(f"Found {len(all_images)} images.")

    random.seed(seed)
    selected = random.sample(all_images, min(num_images, len(all_images)))

    # ── captions ─────────────────────────────────────────────────────────────
    caption_map = {}

    import csv

    for cap_file in list(base.rglob("captions.txt")) + list(base.rglob("*.csv")):
        logger.info(f"Trying caption file: {cap_file}")
        try:
            with open(cap_file, newline="", encoding="utf-8") as f:
                # Sniff the delimiter
                sample = f.read(2048); f.seek(0)
                try:
                    dialect = csv.Sniffer().sniff(sample, delimiters=",\t")
                except csv.Error:
                    dialect = csv.excel   # fallback: comma

                reader = csv.reader(f, dialect)
                header = next(reader, None)

                # Detect column indices
                if header:
                    header_lower = [h.lower().strip() for h in header]
                    img_idx = next((i for i, h in enumerate(header_lower)
                                    if "image" in h or "file" in h), 0)
                    cap_idx = next((i for i, h in enumerate(header_lower)
                                    if "caption" in h or "comment" in h
                                    or "description" in h), 1)
                else:
                    img_idx, cap_idx = 0, 1   # no header: image,caption

                for row in reader:
                    if len(row) <= max(img_idx, cap_idx):
                        continue
                    key = Path(row[img_idx].strip()).name
                    cap = row[cap_idx].strip()
                    if key and cap and key not in caption_map:
                        caption_map[key] = cap

        except Exception as e:
            logger.warning(f"  Could not parse {cap_file}: {e}")
            continue

        if caption_map:
            logger.info(f"  Loaded {len(caption_map)} captions from {cap_file.name}")
            break

    image_paths, captions = [], []
    for p in selected:
        cap = caption_map.get(p.name) or caption_map.get(p.stem) or f"a photo of {p.stem.replace('_',' ')}"
        image_paths.append(str(p))
        captions.append(cap)

    logger.info(f"Using {len(image_paths)} pairs.")
    for i in range(min(3, len(captions))):
        logger.info(f"  [{i}] {Path(image_paths[i]).name} → \"{captions[i][:80]}\"")
    return image_paths, captions



In [ ]:
IMG_TRANSFORM = transforms.Compose([
    transforms.Resize(RESOLUTION, interpolation=transforms.InterpolationMode.BILINEAR),
    transforms.CenterCrop(RESOLUTION),
    transforms.ToTensor(),
    transforms.Normalize([0.5], [0.5]),
])


@torch.no_grad()
def precompute_embeddings(image_paths, captions, device):
    """
    Encode all images (VAE) and captions (CLIP×2 + T5) to tensors,
    store on CPU. Returns lists ready to build EmbeddingDataset.
    """
    weight_dtype = torch.float16   # T4 is fp16 only

    # ── load frozen encoders ─────────────────────────────────────────────────
    logger.info("Loading VAE …")
    vae = AutoencoderKL.from_pretrained(MODEL_ID, subfolder="vae",
                                        torch_dtype=weight_dtype).to(device)
    vae.eval()
    vram_usage("after VAE load")

    logger.info("Loading CLIP encoders …")
    clip_tok_l  = CLIPTokenizer.from_pretrained(MODEL_ID, subfolder="tokenizer")
    clip_tok_g  = CLIPTokenizer.from_pretrained(MODEL_ID, subfolder="tokenizer_2")
    clip_enc_l  = CLIPTextModelWithProjection.from_pretrained(
                        MODEL_ID, subfolder="text_encoder",
                        torch_dtype=weight_dtype).to(device)
    clip_enc_g  = CLIPTextModelWithProjection.from_pretrained(
                        MODEL_ID, subfolder="text_encoder_2",
                        torch_dtype=weight_dtype).to(device)
    clip_enc_l.eval(); clip_enc_g.eval()
    vram_usage("after CLIP load")

    # T5-XXL stays on CPU — too large for T4 alongside the transformer
    logger.info("Loading T5 on CPU (large model — intentionally kept off GPU) …")
    t5_tok = T5TokenizerFast.from_pretrained(MODEL_ID, subfolder="tokenizer_3")
    t5_enc = T5EncoderModel.from_pretrained(MODEL_ID, subfolder="text_encoder_3",
                                             torch_dtype=weight_dtype)   # CPU
    t5_enc.eval()

    all_latents, all_prompt_embeds, all_pooled = [], [], []

    for img_path, caption in tqdm(zip(image_paths, captions),
                                   total=len(image_paths), desc="Pre-computing"):
        #  VAE
        img = Image.open(img_path).convert("RGB")
        pixel = IMG_TRANSFORM(img).unsqueeze(0).to(device, dtype=weight_dtype)
        latent = vae.encode(pixel).latent_dist.sample() * vae.config.scaling_factor
        all_latents.append(latent.squeeze(0).cpu())

        #  CLIP-L
        def clip_encode(tok, enc, text):
            toks = tok(text, padding="max_length", max_length=77,
                        truncation=True, return_tensors="pt").to(device)
            out  = enc(**toks, output_hidden_states=True)
            return out.hidden_states[-2], out.text_embeds   # [1,77,D], [1,D]

        h_l, p_l = clip_encode(clip_tok_l, clip_enc_l, caption)
        h_g, p_g = clip_encode(clip_tok_g, clip_enc_g, caption)
        clip_hidden = torch.cat([h_l, h_g], dim=-1)   # [1,77,2048]

        # T5 (CPU inference, result moved to CPU store)
        t5_toks = t5_tok(caption, padding="max_length", max_length=256,
                          truncation=True, return_tensors="pt")   # stays on CPU
        t5_hidden = t5_enc(**t5_toks).last_hidden_state            # [1,256,4096] CPU

        # pad CLIP to 4096 and concat along sequence dim
        clip_pad = F.pad(clip_hidden.cpu(), (0, 4096 - clip_hidden.shape[-1]))  # [1,77,4096]
        prompt_embeds = torch.cat([clip_pad, t5_hidden], dim=1)    # [1,333,4096]
        pooled_embeds = torch.cat([p_l, p_g], dim=-1)              # [1,2048]

        all_prompt_embeds.append(prompt_embeds.squeeze(0).cpu())
        all_pooled.append(pooled_embeds.squeeze(0).cpu())

    # free all frozen encoders from GPU
    logger.info("Freeing encoders from GPU …")
    free_memory(vae, clip_enc_l, clip_enc_g, t5_enc,
                clip_tok_l, clip_tok_g, t5_tok)
    vram_usage("after encoder cleanup")

    return all_latents, all_prompt_embeds, all_pooled



In [ ]:
def train():
    torch.manual_seed(SEED)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # accelerator
    project_cfg = ProjectConfiguration(project_dir=OUTPUT_DIR,
                                        logging_dir=os.path.join(OUTPUT_DIR, "logs"))
    accelerator = Accelerator(
        gradient_accumulation_steps=GRADIENT_ACCUM,
        mixed_precision=MIXED_PRECISION,
        project_config=project_cfg,
    )

    logger.info(f"Device: {device}")
    vram_usage("start")

    # dataset
    image_paths, captions = load_kaggle_dataset(KAGGLE_DATASET, NUM_IMAGES, SEED)

    # Pre-compute everything so encoders don't compete for VRAM during training
    latents, prompt_embeds, pooled_embeds = precompute_embeddings(image_paths, captions, device)

    dataset    = EmbeddingDataset(latents, prompt_embeds, pooled_embeds)
    dataloader = DataLoader(dataset, batch_size=TRAIN_BATCH_SIZE,
                             shuffle=True, num_workers=0, pin_memory=False)

    os.makedirs(OUTPUT_DIR, exist_ok=True)

    # transformer (trainable)
    logger.info("Loading SD3 transformer …")
    transformer = SD3Transformer2DModel.from_pretrained(
        MODEL_ID, subfolder="transformer",
        torch_dtype=torch.float16,
    )
    transformer.requires_grad_(True)   # full finetune — all params

    if GRADIENT_CKPT:
        transformer.enable_gradient_checkpointing()

    vram_usage("after transformer load")

    #  noise scheduler
    noise_scheduler = FlowMatchEulerDiscreteScheduler.from_pretrained(
        MODEL_ID, subfolder="scheduler"
    )

    # optimizer
    if USE_8BIT_ADAM:
        try:
            import bitsandbytes as bnb
            optimizer = bnb.optim.AdamW8bit(
                transformer.parameters(), lr=LEARNING_RATE,
                betas=(0.9, 0.999), weight_decay=1e-2, eps=1e-8,
            )
            logger.info("Using 8-bit AdamW (bitsandbytes)")
        except ImportError:
            logger.warning("bitsandbytes not found — falling back to fp32 AdamW")
            optimizer = torch.optim.AdamW(
                transformer.parameters(), lr=LEARNING_RATE,
                betas=(0.9, 0.999), weight_decay=1e-2, eps=1e-8,
            )
    else:
        optimizer = torch.optim.AdamW(
            transformer.parameters(), lr=LEARNING_RATE,
            betas=(0.9, 0.999), weight_decay=1e-2, eps=1e-8,
        )

    num_update_steps = math.ceil(len(dataloader) / GRADIENT_ACCUM) * NUM_EPOCHS
    lr_scheduler = get_cosine_schedule_with_warmup(
        optimizer,
        num_warmup_steps=LR_WARMUP_STEPS,
        num_training_steps=num_update_steps,
    )

    transformer, optimizer, dataloader, lr_scheduler = accelerator.prepare(
        transformer, optimizer, dataloader, lr_scheduler
    )
    vram_usage("after accelerate prepare")

    # training loop
    logger.info("━━━━━  Starting Training  ━━━━━")
    logger.info(f"  Model            : {MODEL_ID}")
    logger.info(f"  Dataset size     : {len(dataset)}")
    logger.info(f"  Resolution       : {RESOLUTION}×{RESOLUTION}")
    logger.info(f"  Batch / Accum    : {TRAIN_BATCH_SIZE} / {GRADIENT_ACCUM}")
    logger.info(f"  Epochs           : {NUM_EPOCHS}")
    logger.info(f"  Learning rate    : {LEARNING_RATE}")
    logger.info(f"  Mixed precision  : {MIXED_PRECISION}")
    logger.info(f"  Gradient ckpt    : {GRADIENT_CKPT}")
    logger.info(f"  8-bit Adam       : {USE_8BIT_ADAM}")

    global_step = 0

    for epoch in range(NUM_EPOCHS):
        transformer.train()
        epoch_loss = 0.0

        pbar = tqdm(total=len(dataloader), desc=f"Epoch {epoch+1}/{NUM_EPOCHS}")

        for step, batch in enumerate(dataloader):
            with accelerator.accumulate(transformer):

                # move pre-computed tensors to GPU
                latents_b       = batch["latents"].to(device, dtype=torch.float16)
                prompt_embeds_b = batch["prompt_embeds"].to(device, dtype=torch.float16)
                pooled_embeds_b = batch["pooled_embeds"].to(device, dtype=torch.float16)

                bsz = latents_b.shape[0]

                # timestep + noise
                timesteps = torch.randint(
                    0, noise_scheduler.config.num_train_timesteps,
                    (bsz,), device=device,
                ).long()

                noise         = torch.randn_like(latents_b)
                # Manual linear interpolation — avoids scale_noise() which
                # requires scheduler.set_timesteps() (inference-only call).
                noisy_latents = add_flow_noise(
                    latents_b, noise, timesteps,
                    num_train_timesteps=noise_scheduler.config.num_train_timesteps,
                )

                # forward
                model_pred = transformer(
                    hidden_states         = noisy_latents,
                    timestep              = timesteps,
                    encoder_hidden_states = prompt_embeds_b,
                    pooled_projections    = pooled_embeds_b,
                    return_dict           = False,
                )[0]

                # Flow-matching target: velocity field  v = noise − x_0
                # The model predicts v_t, we regress against the ground-truth v.
                target = noise - latents_b
                loss   = F.mse_loss(model_pred.float(), target.float(), reduction="mean")

                epoch_loss += loss.detach().item()
                accelerator.backward(loss)

                if accelerator.sync_gradients:
                    accelerator.clip_grad_norm_(transformer.parameters(), MAX_GRAD_NORM)

                optimizer.step()
                lr_scheduler.step()
                optimizer.zero_grad(set_to_none=True)   # set_to_none saves a tiny bit extra

            if accelerator.sync_gradients:
                global_step += 1

            pbar.update(1)
            pbar.set_postfix(loss=f"{loss.detach().item():.4f}")

        pbar.close()
        avg = epoch_loss / len(dataloader)
        logger.info(f"Epoch {epoch+1}/{NUM_EPOCHS} — avg loss: {avg:.4f}")
        vram_usage(f"end epoch {epoch+1}")

        # checkpoint
        if (epoch + 1) % SAVE_EVERY_N_EPOCHS == 0:
            ckpt = os.path.join(OUTPUT_DIR, f"checkpoint-epoch-{epoch+1}")
            accelerator.unwrap_model(transformer).save_pretrained(ckpt)
            logger.info(f"Saved checkpoint → {ckpt}")

    # save full pipeline
    accelerator.wait_for_everyone()
    logger.info("Saving final pipeline …")
    pipeline = StableDiffusion3Pipeline.from_pretrained(
        MODEL_ID,
        transformer=accelerator.unwrap_model(transformer),
        torch_dtype=torch.float16,
    )
    pipeline.save_pretrained(OUTPUT_DIR)
    logger.info(f"Done! Pipeline saved to {OUTPUT_DIR}")
    accelerator.end_training()

In [ ]:
def generate_samples(prompt: str, num_images: int = 2, out_dir: str = "/content/samples"):
    """
    Load the fine-tuned pipeline and generate samples.
    Runs with enable_model_cpu_offload() to stay within 16 GB.
    """
    import matplotlib.pyplot as plt

    pipeline = StableDiffusion3Pipeline.from_pretrained(
        OUTPUT_DIR, torch_dtype=torch.float16
    )
    # CPU offload: moves model layers to GPU one at a time — safe on T4
    pipeline.enable_model_cpu_offload()

    os.makedirs(out_dir, exist_ok=True)
    images = pipeline(
        prompt,
        num_images_per_prompt=num_images,
        num_inference_steps=28,
        height=RESOLUTION,
        width=RESOLUTION,
    ).images

    cols = num_images
    fig, axes = plt.subplots(1, cols, figsize=(4 * cols, 4))
    if cols == 1:
        axes = [axes]
    for i, (img, ax) in enumerate(zip(images, axes)):
        ax.imshow(img); ax.axis("off"); ax.set_title(f"#{i+1}")
        img.save(os.path.join(out_dir, f"sample_{i+1}.png"))

    plt.suptitle(prompt, wrap=True)
    plt.tight_layout()
    plt.savefig(os.path.join(out_dir, "grid.png"))
    plt.show()
    print(f"Saved {num_images} image(s) to {out_dir}")


In [ ]:
if __name__ == "__main__":
    train()

INFO:__main__:Device: cuda
INFO:__main__:Downloading Kaggle dataset: adityajn105/flickr8k


[VRAM start] allocated 9.78 GB | reserved 10.25 GB
Using Colab cache for faster access to the 'flickr8k' dataset.


INFO:__main__:Found 8091 images.
INFO:__main__:Trying caption file: /kaggle/input/flickr8k/captions.txt
INFO:__main__:  Loaded 8091 captions from captions.txt
INFO:__main__:Using 2 pairs.
INFO:__main__:  [0] 3354489242_dd529ffa1f.jpg → "A goalie tries to block the puck in a hockey game ."
INFO:__main__:  [1] 2071309418_1d7580b0f0.jpg → "A white dog wearing a christmas reindeer headband plays with a brown dog in the "
INFO:__main__:Loading VAE …
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_validators.py:202: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(
INFO:httpx:HTTP Request: HEAD https://huggingface.co/stabilityai/stable-diffusion-3-medium-diffusers/resolve/main/vae/config.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/stabilityai/stable-diffusion-3-medium-diffusers/resolve/ea42f8cef0f178587cf766dc8129abd3

[VRAM after VAE load] allocated 9.95 GB | reserved 10.31 GB


INFO:httpx:HTTP Request: HEAD https://huggingface.co/stabilityai/stable-diffusion-3-medium-diffusers/resolve/main/tokenizer/tokenizer_config.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/models/stabilityai/stable-diffusion-3-medium-diffusers/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/models/stabilityai/stable-diffusion-3-medium-diffusers/tree/main?recursive=true&expand=false "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/stabilityai/stable-diffusion-3-medium-diffusers/resolve/main/tokenizer_2/tokenizer_config.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/models/stabilityai/stable-diffusion-3-medium-diffusers/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/models/stabilityai/stable-diffusion-3-medium-

...

[VRAM after transformer load] allocated 9.79 GB | reserved 10.98 GB


INFO:httpx:HTTP Request: HEAD https://huggingface.co/stabilityai/stable-diffusion-3-medium-diffusers/resolve/main/scheduler/scheduler_config.json "HTTP/1.1 200 OK"
INFO:__main__:Using 8-bit AdamW (bitsandbytes)
INFO:__main__:━━━━━  Starting Training  ━━━━━
INFO:__main__:  Model            : stabilityai/stable-diffusion-3-medium-diffusers
INFO:__main__:  Dataset size     : 2
INFO:__main__:  Resolution       : 256×256
INFO:__main__:  Batch / Accum    : 1 / 8
INFO:__main__:  Epochs           : 10
INFO:__main__:  Learning rate    : 5e-06
INFO:__main__:  Mixed precision  : fp16
INFO:__main__:  Gradient ckpt    : True
INFO:__main__:  8-bit Adam       : True


[VRAM after accelerate prepare] allocated 14.03 GB | reserved 14.58 GB


Epoch 1/10:   0%|          | 0/2 [00:00<?, ?it/s]

OutOfMemoryError: CUDA out of memory. Tried to allocate 18.00 MiB. GPU 0 has a total capacity of 14.56 GiB of which 1.81 MiB is free. Including non-PyTorch memory, this process has 14.56 GiB memory in use. Of the allocated memory 13.91 GiB is allocated by PyTorch, and 519.07 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)